# 🧬 02 — Extract Activations & Find Refusal Direction (Google Colab)

**Tujuan:** Ekstrak aktivasi residual stream harmful vs harmless prompts, lalu hitung refusal direction.

**Kenapa di Colab:** Notebook ini butuh forward pass dengan cache aktivasi — VRAM lokal 4GB tidak cukup.

**⚠️ Runtime:** `Runtime → Change runtime type → **T4 GPU**` (bukan TPU!)

**Input (dari Drive):**
- `data/harmful_prompts.csv`
- `data/harmless_prompts.csv`

**Output (ke Drive):**
- `models/abliterated/refusal_dirs.safetensors`
- `models/abliterated/refusal_dirs.pth`

---
**Cara pakai:** Ubah `PROJECT_DIR` → `Runtime → Run all`

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup path & install dependencies

**UBAH `PROJECT_DIR`** sesuai folder proyek kamu di Drive.

In [ ]:
import os
PROJECT_DIR = os.environ.get('COLAB_DRIVE_PATH', '/content/drive/MyDrive/llama32-1b-llm-abliteration')
os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)

print(f'Project: {PROJECT_DIR}')
print(f'Content: {os.listdir(".")}')

In [ ]:
%pip install -r "{PROJECT_DIR}/requirements.txt"

import gc
import torch
from collections import defaultdict
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, utils
from safetensors.torch import save_file, load_file

torch.set_grad_enabled(False)
print('Dependencies installed')

## 3. Login HuggingFace (jika model gated)

Llama-3.2-1B-Instruct mungkin butuh autentikasi. Jalankan cell ini kalau muncul error 401/403.

**Kalo nggak error, skip aja.**

In [ ]:
# Uncomment kalau perlu login:
# from huggingface_hub import login
# import os
# login(token=os.getenv('HF_TOKEN'))

## 4. Import konfigurasi lokal

**Override `BATCH_SIZE_EXTRACT` jadi 8** — T4 16GB kuat buat batch lebih besar.

In [ ]:
from src.config import DEFAULT_MODEL, DEVICE, N_INST_TRAIN, ACTIVATION_LAYERS, ABLITERATED_DIR, SEED
from src.data_loader import get_harmful_instructions, get_harmless_instructions
from src.generation_utils import tokenize_instructions_fn, clear_memory

# Override: T4 16GB kuat buat batch lebih besar
BATCH_SIZE_EXTRACT = 8   # vs 4 di lokal

torch.manual_seed(SEED)
print(f'Device: {DEVICE}')
print(f'Model: {DEFAULT_MODEL}')
print(f'GPU: {torch.cuda.get_device_properties(0).name}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print(f'N_INST_TRAIN: {N_INST_TRAIN}, BATCH_SIZE_EXTRACT: {BATCH_SIZE_EXTRACT}')

## 5. Load model

Model akan di-download ke cache Colab (`~/.cache/huggingface/`) — lebih cepat daripada Drive.

In [ ]:
model = HookedTransformer.from_pretrained_no_processing(
    DEFAULT_MODEL,
    dtype=torch.bfloat16,
    default_padding_side='left',
    device=DEVICE,
)
model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f'Model: {model.cfg.model_name}')
print(f'n_layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## 6. Load data

In [ ]:
harmful = get_harmful_instructions()[:N_INST_TRAIN]
harmless = get_harmless_instructions()[:N_INST_TRAIN]
print(f'Harmful: {len(harmful)}, Harmless: {len(harmless)}')

## 7. Tokenize

In [ ]:
toks = tokenize_instructions_fn(model, instructions=harmful + harmless)
harmful_toks, harmless_toks = toks.split(N_INST_TRAIN)
print(f'Harmful tokens: {harmful_toks.shape}')
print(f'Harmless tokens: {harmless_toks.shape}')

## 8. Ekstraksi aktivasi

- Batch 8 prompt per forward pass
- Cache langsung dipindah ke CPU biar VRAM lega
- Cleanup tiap batch

In [ ]:
harm_cache = defaultdict(list)
harmless_cache = defaultdict(list)

batch_size = BATCH_SIZE_EXTRACT
num_batches = (N_INST_TRAIN + batch_size - 1) // batch_size

for i in tqdm(range(num_batches)):
    s = i * batch_size
    e = min(N_INST_TRAIN, s + batch_size)

    harmful_logits, harmful_c = model.run_with_cache(
        harmful_toks[s:e].to(DEVICE),
        names_filter=lambda hook_name: 'resid' in hook_name,
        device='cpu',            # cache ke CPU, bukan GPU
        reset_hooks_end=True,
    )
    harmless_logits, harmless_c = model.run_with_cache(
        harmless_toks[s:e].to(DEVICE),
        names_filter=lambda hook_name: 'resid' in hook_name,
        device='cpu',
        reset_hooks_end=True,
    )

    for key in harmful_c:
        harm_cache[key].append(harmful_c[key])
        harmless_cache[key].append(harmless_c[key])

    del harmful_logits, harmless_logits, harmful_c, harmless_c
    clear_memory()

# Gabung batch
harm = {k: torch.cat(v) for k, v in harm_cache.items()}
harmless = {k: torch.cat(v) for k, v in harmless_cache.items()}

sample_key = utils.get_act_name('resid_pre', 0)
print(f'Harmful activations: {harm[sample_key].shape}')
print(f'Harmless activations: {harmless[sample_key].shape}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## 9. Hitung refusal direction

In [ ]:
def get_act_idx(cache_dict, act_name, layer):
    return cache_dict[utils.get_act_name(act_name, layer)]

activation_refusals = {k: [] for k in ACTIVATION_LAYERS}

for layer_num in range(1, model.cfg.n_layers):
    pos = -1  # token terakhir
    for layer in ACTIVATION_LAYERS:
        harmful_mean = get_act_idx(harm, layer, layer_num)[:, pos, :].mean(dim=0)
        harmless_mean = get_act_idx(harmless, layer, layer_num)[:, pos, :].mean(dim=0)
        refusal_dir = harmful_mean - harmless_mean
        refusal_dir = refusal_dir / refusal_dir.norm()
        activation_refusals[layer].append(refusal_dir)

print('Refusal directions computed!')
for layer in ACTIVATION_LAYERS:
    print(f'  {layer}: {len(activation_refusals[layer])} directions')

## 10. Simpan ke Drive

In [ ]:
ABLITERATED_DIR.mkdir(parents=True, exist_ok=True)

# Safetensors (utama)
refusal_tensors = {}
for layer_name, dir_list in activation_refusals.items():
    for idx, dir_tensor in enumerate(dir_list):
        refusal_tensors[f'{layer_name}.layer_{idx+1}'] = dir_tensor

save_file(refusal_tensors, str(ABLITERATED_DIR / 'refusal_dirs.safetensors'))
print(f'Saved: {ABLITERATED_DIR / "refusal_dirs.safetensors"}')
print(f'   {len(refusal_tensors)} tensors')

# .pth cadangan
torch.save(activation_refusals, str(ABLITERATED_DIR / 'refusal_dirs.pth'))
print(f'Backup: {ABLITERATED_DIR / "refusal_dirs.pth"}')

## 11. Verifikasi

In [ ]:
loaded = load_file(str(ABLITERATED_DIR / 'refusal_dirs.safetensors'))
print(f'Verified: {len(loaded)} keys')
for k, v in list(loaded.items())[:3]:
    print(f'  {k}: shape={v.shape}')

## 12. Bersihin memory (opsional)

Biar GPU lega kalau lanjut notebook lain di runtime yang sama.

In [ ]:
del model, harm, harmless, harmful_toks, harmless_toks
clear_memory()
print('Cleanup done')
print(f'VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

---
**Next:** Lanjut ke `03_screening_directions.ipynb` di lokal — hasil sudah ada di Drive.

**Estimasi waktu:** ~5-10 menit (download model + 64 batch forward pass)